# Structured streaming Demo

In [1]:
# Imports cell

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType

from pyspark.sql.functions import window, col, count

# Convert timestamp (Unix epoch in seconds) to timestamp type
from pyspark.sql.functions import from_unixtime




In [2]:
# initialize the spark session
spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0") \
    .getOrCreate()


In [3]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()


In [4]:
# Define a simplified schema - just the key fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")

In [5]:
from pyspark.sql import functions as F

# Préparer la date et la variation de taille de chaque page.
edits_df = (
    parsed_df
    .withColumn("event_time", F.col("timestamp").cast("timestamp"))
    .withColumn(
        "size_delta",
        F.col("length.new") - F.col("length.old")
    )
)

def aggregate_edits(window_duration, slide_duration):
    return (
        edits_df
        .groupBy(
            F.window("event_time", window_duration, slide_duration),
            "bot"
        )
        .agg(
            F.count("*").alias("edit_count"),
            F.count("size_delta").alias("edits_with_size"),
            F.round(F.avg("size_delta"), 2).alias("avg_size_delta"),
            F.sum("size_delta").alias("net_size_delta"),
            F.max(F.abs(F.col("size_delta"))).alias("largest_edit")
        )
        .select(
            F.col("window.start").alias("window_start"),
            F.col("window.end").alias("window_end"),
            "bot",
            "edit_count",
            "edits_with_size",
            "avg_size_delta",
            "net_size_delta",
            "largest_edit"
        )
    )

# Fenêtres fixes de 1 minute, sans chevauchement.
tumbling_df = aggregate_edits("1 minute", "1 minute")

# Fenêtres de 2 minutes, démarrant toutes les 30 secondes.
sliding_df = aggregate_edits("2 minutes", "30 seconds")

In [6]:
queries = []

for name, dataframe in [
    ("tumbling_edits", tumbling_df),
    ("sliding_edits", sliding_df),
]:
    query = (
        dataframe.writeStream
        .queryName(name)
        .format("console")
        .outputMode("complete")
        .option("truncate", False)
        .option("numRows", 50)
        .trigger(processingTime="10 seconds")
        .start()
    )
    queries.append(query)

In [7]:
for query in spark.streams.active:
    query.stop()